# Attention Entropy Dynamics

How attention entropy evolves across layers, positions, and heads:
layer trends, position profiles, sharpening, and diversity.

## Why This Matters

Attention entropy dynamics reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_entropy_dynamics import (
    entropy_by_layer, entropy_by_position,
    entropy_head_evolution, entropy_sharpening_profile,
    entropy_diversity_across_heads,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Entropy by Layer

Average attention entropy across heads for each layer.

In [ ]:
result = entropy_by_layer(model, tokens)
for p in result['per_layer']:
    bar = '█' * int(p['normalized_entropy'] * 20)
    print(f"  Layer {p['layer']}: H={p['mean_entropy']:.4f} "
          f"(norm={p['normalized_entropy']:.3f}), "
          f"range=[{p['min_head_entropy']:.3f}, {p['max_head_entropy']:.3f}] {bar}")

## Entropy by Position

How does entropy vary across query positions? Early positions
have less context to attend to.

In [ ]:
result = entropy_by_position(model, tokens, layer=0)
for p in result['per_position']:
    bar = '█' * int(p['normalized_entropy'] * 20)
    print(f"  Pos {p['position']}: H={p['mean_entropy']:.4f}/{p['max_possible']:.4f} "
          f"(norm={p['normalized_entropy']:.3f}) {bar}")

## Head Evolution

How does a specific head index change across layers?

In [ ]:
for head_idx in range(4):
    result = entropy_head_evolution(model, tokens, head=head_idx)
    ents = [p['mean_entropy'] for p in result['per_layer']]
    trend = '↓' if ents[-1] < ents[0] else '↑'
    print(f"  Head {head_idx}: {' → '.join(f'{e:.3f}' for e in ents)} {trend}")

## Sharpening Profile

Does attention get sharper (lower entropy) in deeper layers?

In [ ]:
result = entropy_sharpening_profile(model, tokens)
print(f"Sharpening: {result['is_sharpening']}")
print(f"First half mean: {result['first_half_mean']:.4f}")
print(f"Second half mean: {result['second_half_mean']:.4f}")
print(f"\nPer-layer: {', '.join(f'{e:.3f}' for e in result['layer_entropies'])}")

## Diversity Across Heads

Are heads within a layer at different entropy levels (= diverse roles)?

In [ ]:
result = entropy_diversity_across_heads(model, tokens)
for p in result['per_layer']:
    flag = ' [DIVERSE]' if p['is_diverse'] else ''
    print(f"  Layer {p['layer']}: mean_H={p['mean_entropy']:.4f}, "
          f"std={p['std_entropy']:.4f}, range={p['entropy_range']:.4f}{flag}")